# NEXORA — Experiments
**Machine Translation using Sequence-to-Sequence Networks with Attention-Based Alignment**

This notebook walks through the same pipeline as `build_pipeline.py`, cell by cell. It uses the **Samanantar** corpus (AI4Bharat) and reports only measured numbers.

In [ ]:
# imports
import sys, json
from pathlib import Path
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from src.multilingual import light_tokenize

## 1. Dataset & preprocessing

In [ ]:
from src.preprocessing import load_raw_subset, clean_text, filter_valid_sentence, LID_LANGS

raw = load_raw_subset()
print("raw rows:", len(raw))
print(raw.head(3))

In [ ]:
records = []
for lang in LID_LANGS:
    texts = raw.loc[raw["language"] == lang, "text"]
    cleaned = [clean_text(t, lang) for t in texts]
    kept = [(t, lang) for t in cleaned if filter_valid_sentence(t, lang)]
    records.extend(kept)
lid = pd.DataFrame(records, columns=["text", "language"]).drop_duplicates().reset_index(drop=True)
lid["language"].value_counts()

## 2. EDA quick look

In [ ]:
lid["text"].str.split().str.len().describe()

In [ ]:
from src.eda import compute_dataset_stats
compute_dataset_stats(lid, {}).round(2)

## 3. Feature engineering + classical models

In [ ]:
from src.features import build_features
feats = build_features(lid)
print("feature spaces:", list(feats["fitted"].keys()))

In [ ]:
from src.classical_models import train_all
payload = train_all(feats)
payload["results"]

## 4. Experimental Seq2Seq + Attention

In [ ]:
from src.preprocessing import DATASET_NAME
from datasets import load_dataset

rows = []
for i, row in enumerate(load_dataset(DATASET_NAME, "hi", split="train", streaming=True)):
    if i >= 30000:
        break
    rows.append((row["src"], row["tgt"]))
pair = pd.DataFrame(rows, columns=["src", "tgt"])
print(len(pair))

In [ ]:
from src.seq2seq import Vocab, train_model
# train_model saves artifacts under models/seq2seq and returns metrics
meta, model, vs, vt = train_model(pair, Path("../models/seq2seq"), sample=20000, epochs=2, batch_size=32)
print("BLEU:", meta["bleu"])
print("history:", meta["history"])

## 5. Attention heatmap

In [ ]:
import torch
from src.seq2seq import Seq2SeqAttention
m = Seq2SeqAttention(len(vs), len(vt), 128, 128)
m.load_state_dict(torch.load("../models/seq2seq/seq2seq_attention.pt", map_location="cpu"))
preds, attn = m.translate(["How are you ?"], vs, vt, max_len=24)
print(preds)
from src.attention import tokens_heatmap
tokens_heatmap(attn[0], light_tokenize("How are you ?"), light_tokenize(preds[0]))

## 6. Production translation (M2M100)

In [ ]:
from src.translation import get_translator
tr = get_translator()
r = tr.translate("How are you?", "en", "hi")
print(r.target)